# 04. The LSTM, and how it is kept from seeing the future

Milestone 4: `lstm_log`, an LSTM over the 22 sessions `t-21..t` with HAR's regressors beside its last
hidden state, on both tracks: the VIX level at h = 1, 5, 21 and `rv21` at h = 21. Same folds, same
origins, same losses as every M2/M3 model.

At every training cutoff c, with H = 21 the longest horizon:

```
usable origins      tau <= c - H                 (every target dated <= c: rule L2)
validation          the last 252 of them: v0 .. c - H
training            tau <= v0 - H                (the last training target is dated v0)
scaler              mean and sd of the training slice only
forecast            mu = a_t + network output   (a_t = ln VIX_t or ln rv21_t)
                    yhat = exp(mu + s2/2)        s2 = validation MSE of the seed-mean mu
```

The full run (after `garch`, in the same code state):

```
uv run python -m ivf.models lstm            # real data, local only
uv run python -m ivf.models --synthetic lstm --print-metrics
```

This notebook runs the last `IVF_NOTEBOOK_FOLDS` dev folds (default 4) with the committed settings
(`IVF_NOTEBOOK_QUICK=1`: one seed, a tiny network). **Licensing:** real-data numbers stay on your
machine (ADR 0001 D2).

*This file is generated by `tools/build_notebooks.py`. Edit the spec there.*

In [ ]:
import dataclasses
import json
import os
import warnings

import numpy as np
import pandas as pd

from ivf import config, protocol
from ivf.data import build_interim, calendar, check, synthetic
from ivf.evaluation import walkforward
from ivf.features import build
from ivf.models import baselines, lstm, registry, rv21
from ivf.models.cli import member_table

warnings.filterwarnings("ignore", category=FutureWarning)


def load():
    """(mode, protocol, iv, price, build_report): real if available, else synthetic."""
    if os.environ.get("IVF_NOTEBOOK_DATA") != "synthetic":
        try:
            proto = protocol.load_protocol(protocol.protocol_path())
            proto.require_data_end()
            iv, price, report = build_interim.read_interim(config.data_layout().root)
            return "real", proto, iv, price, report
        except (protocol.ProtocolError, build_interim.InterimError) as why:
            print("no real data:", why)
    proto = protocol.load_protocol(protocol.protocol_path(synthetic=True))
    iv, price = synthetic.synthetic_market(proto)
    report = {"files": [{"symbol": s, "fetched_at": "2026-10-27T12:00:00.000000Z", "duplicates_dropped": 0, "duplicate_dates": []}
                        for s in list(iv["symbol"].unique()) + ["^GSPC"]]}
    return "synthetic", proto, iv, price, report


MODE, PROTO, IV, PRICE, REPORT = load()
if not check.frozen_check(PROTO, IV, PRICE, REPORT).ok:
    raise RuntimeError("check reports VIOLATED: run `uv run python -m ivf.data check` and fix the data first")
CAL = calendar.master_calendar(PROTO)
CALENDAR_DATES = list(CAL["date"])
DEV = [f for f in walkforward.fold_schedule(CALENDAR_DATES, PROTO.dev_start, PROTO.holdout_start, PROTO.fold_len, PROTO.data_end) if f.period == "dev"]
DATES, _, IV_PRE, PRICE_PRE = walkforward.pre_holdout_inputs(CAL, IV, PRICE, PROTO.holdout_start)
VIX = build.align(DATES, IV_PRE, "symbol", "VIX")
INPUTS = rv21.Rv21Inputs(build.log_returns(build.align(DATES, PRICE_PRE, "ticker", "^GSPC")), VIX)
N_FOLDS = int(os.environ.get("IVF_NOTEBOOK_FOLDS", "4"))
SETTINGS = registry.load_lstm_settings()
if os.environ.get("IVF_NOTEBOOK_QUICK") == "1":
    SETTINGS = dataclasses.replace(SETTINGS, members=1, hidden=4, max_epochs=2)
print("data:", MODE.upper(), "| holdout_start", PROTO.holdout_start, "| sessions readable", len(DATES), "| last", DATES[-1])
print("settings:", SETTINGS)

## 1. The slices, by hand

At c = 100 with H = 21 and 10 validation sessions: the last usable origin is 79 (its target is dated
100 = c), validation is 70..79, and training ends at 49, whose 21-session target is dated 70, the first
validation origin. Nothing the validation slice is scored on was trained on.

In [ ]:
train, val = lstm.split(100, 21, 10)
print(f"training {train[0]}..{train[-1]}, validation {val[0]}..{val[-1]}, gap {val[0] - train[-1]} sessions")
c = DEV[-N_FOLDS].cutoff_idx
train, val = lstm.split(c, 21, SETTINGS.val_sessions)
print(f"at cutoff {DATES[c]}: training origins up to {DATES[train[-1]]}, validation {DATES[val[0]]} .. {DATES[val[-1]]}")

## 2. The VIX level track

`lstm_log` beside the random walk and `har_log` (the LSTM's head sees HAR's regressors, so HAR is the
model it has to beat). With `refit_every` = 252 sessions, the network trained at the first fold here is
applied unchanged at the next three.

In [ ]:
SEED = PROTO.document["seed"]  # protocol.seed: member m at cutoff d gets SeedSequence(SEED, (key32, yyyymmdd(d), m))
VIX_MODEL = lstm.LstmLog(SETTINGS, SEED, DATES)
VIX_RUN = walkforward.run_symbol("VIX", "vix_level", VIX, DATES, CALENDAR_DATES, DEV[-N_FOLDS:],
                                 [baselines.RandomWalk(), baselines.Har(), VIX_MODEL], PROTO.horizons, PROTO.holdout_start, 504)
for _, row in VIX_RUN.fits[VIX_RUN.fits.model_id == "lstm_log"].iterrows():
    p = json.loads(row.params_json)
    m = p["members"][row.member - 1]
    print(f"cutoff {row.cutoff_date} seed {row.seed}: refit {p['refit']}, trained at {DATES[p['trained_cutoff']]}, "
          f"epochs {m['epochs_run']} (best {m['best_epoch']}), n_train {row.n_train}")
VM = walkforward.metric_table(VIX_RUN.forecasts, PROTO.holdout_start, PROTO.document["min_cell_n"], {"har_log", "lstm_log"})
print(VM[VM.forecast == "mean"].pivot_table(index=["horizon", "model_id"], values=["rmse", "qlike"]).round(4))

## 3. The rv21 track

Inputs: the daily returns of the window, and ln VIX, ln rv5 and ln rv21 at t. Every target window of
the training and validation slices ends by the cutoff.

In [ ]:
RV_MODEL = lstm.LstmRv(SETTINGS, SEED, DATES)
RV_RUN = walkforward.run_rv21(INPUTS, DATES, CALENDAR_DATES, DEV[-N_FOLDS:],
                              [rv21.RandomWalkRv(), rv21.HarRv(), rv21.VixImplied(), RV_MODEL], PROTO.holdout_start, 504)
RM = walkforward.metric_table(RV_RUN.forecasts, PROTO.holdout_start, PROTO.document["min_cell_n"], {"lstm_log"})
print(RM[RM.forecast == "mean"].set_index("model_id")[["n", "rmse", "qlike", "bias"]].round(4))

## 4. The seeds

Each seed is a full network. The forecast averages their log forecasts; `forecast_member` keeps each.
The spread across seeds is per forecast; whether a model ranking could be seed luck is a question
about each member's own loss, so compare every member's RMSE with the other models' (on the synthetic
market all five lose to HAR; amended M4, review A).

In [ ]:
FC = pd.concat([VIX_RUN.forecasts, RV_RUN.forecasts], ignore_index=True)
FC = FC[FC.model_id == "lstm_log"]
MEMBERS = member_table(FC)
spread = MEMBERS.groupby(["target_id", "horizon", "origin_date"])["yhat"].std(ddof=0)
rel = spread / FC.set_index(["target_id", "horizon", "origin_date"])["yhat"]
print(f"{len(MEMBERS)} member rows; seed sd / yhat: median {rel.median():.4f}, max {rel.max():.4f}")
SCORED = walkforward.scored(FC, PROTO.holdout_start)[["target_id", "horizon", "origin_date", "y"]]
PER = MEMBERS.merge(SCORED, on=["target_id", "horizon", "origin_date"])
PER = PER.assign(se=(PER["y"] - PER["yhat"]) ** 2).groupby(["target_id", "horizon", "member"])["se"].mean() ** 0.5
print("RMSE of each member:")
print(PER.unstack("member").round(4))

## 5. Same inputs, same bytes

One thread, deterministic algorithms, a seeded generator for the batch order: two fits on the same
data give the same weights, bit for bit.

In [ ]:
a = lstm.LstmLog(dataclasses.replace(SETTINGS, members=1, max_epochs=1), SEED, DATES).fit(VIX[: c + 1], PROTO.horizons, None)
b = lstm.LstmLog(dataclasses.replace(SETTINGS, members=1, max_epochs=1), SEED, DATES).fit(VIX[: c + 1], PROTO.horizons, None)
print("weights sha256 equal:", a.params["members"][0]["weights_sha256"] == b.params["members"][0]["weights_sha256"])